In [1]:
import pickle
import numpy as np
import pandas as pd
import os

# 

In [3]:
def load_scores(path):
    print("loading scores:",path)
    with open(path, 'rb') as handle:
        scores = pickle.load(handle)
    return scores
def build_analysis_dataframe(datasets, cf_methods, models=["mlp"], path_root="plots"):
    """
    Build a dataframe for statistical analysis of selective classification results.
    
    Parameters:
    datasets: list of dataset names
    cf_methods: list of counterfactual methods
    models: list of model names (default: ["mlp"])
    path_root: root path to look for data files
    
    Returns:
    pandas DataFrame with columns:
    - dataset: name of the dataset
    - cf_method: name of the counterfactual method
    - model: name of the ML model
    - selective_classifier: name of the selective classifier
    - metric: name of the metric (rejected_by_coverage, classification_quality, etc.)
    - target_coverage: target coverage rate
    - value: the performance value
    - distance_type: type of distance (for CFDistRejector)
    - aggregation: min, max, or mean (for CFDistRejector)
    - distribution: distribution used (normal or gamma)
    """
    results = []
    
    for dt in datasets:
        dt_path = os.path.join(path_root, dt)
        
        for method in cf_methods:
            method_path = os.path.join(dt_path, method)
            
            name_f = "_".join(["selective_results", dt, method]) + ".pkl"
            path = os.path.join(method_path, name_f)
            
            try:
                scores = load_scores(path)
                
                # Process dataframes (contains performance by target coverage)
                for model in models:
                    if model in scores["dataframes"]:
                        df = scores["dataframes"][model]
                        
                        # Get original accuracy for reference
                        original_acc = scores["all_original_scores"][model]
                        
                        # Process each selective classifier
                        for classifier_name, row in df.iterrows():
                            # Parse classifier information
                            is_baseline = "_" not in classifier_name
                            
                            if is_baseline:
                                distance_type = "N/A"
                                aggregation = "N/A"
                                distribution = "N/A"
                            else:
                                parts = classifier_name.split("_")
                                distance_type = parts[1] if len(parts) > 1 else "N/A"
                                aggregation = parts[2] if len(parts) > 2 else "N/A"
                                distribution = "gamma" if len(parts) > 3 else "normal"
                            
                            # Add each target coverage as a separate row
                            for col in df.columns:
                                target_coverage = float(col)
                                value = row[col]
                                
                                results.append({
                                    "dataset": dt,
                                    "cf_method": method,
                                    "model": model,
                                    "selective_classifier": classifier_name,
                                    "is_baseline": is_baseline,
                                    "metric": "accuracy_by_coverage",
                                    "target_coverage": target_coverage,
                                    "value": value,
                                    "distance_type": distance_type,
                                    "aggregation": aggregation,
                                    "distribution": distribution,
                                    "original_accuracy": original_acc
                                })
                
                # Process metric dictionaries
                for metric_name, metric_dict in scores["metric_dicts"].items():
                    for model in models:
                        if model in metric_dict:
                            for classifier_name, value in metric_dict[model].items():
                                # Parse classifier information (same as above)
                                is_baseline = "_" not in classifier_name
                                
                                if is_baseline:
                                    distance_type = "N/A"
                                    aggregation = "N/A"
                                    distribution = "N/A"
                                else:
                                    parts = classifier_name.split("_")
                                    distance_type = parts[1] if len(parts) > 1 else "N/A"
                                    aggregation = parts[2] if len(parts) > 2 else "N/A"
                                    distribution = "gamma" if len(parts) > 3 else "normal"
                                
                                results.append({
                                    "dataset": dt,
                                    "cf_method": method,
                                    "model": model,
                                    "selective_classifier": classifier_name,
                                    "is_baseline": is_baseline,
                                    "metric": metric_name,
                                    "target_coverage": None,  # Not applicable for these metrics
                                    "value": value,
                                    "distance_type": distance_type,
                                    "aggregation": aggregation,
                                    "distribution": distribution,
                                    "original_accuracy": scores["all_original_scores"][model]
                                })
            except Exception as e:
                print(f"Error processing {path}: {e}")
    
    return pd.DataFrame(results)


In [ ]:
# load_scores
build_analysis_dataframe(datasets= ["adult48k","german_credit"],
            cf_methods=["lore","dice","ils"],
            models=["mlp"],
            path_root="plots_for_latex")

TypeError: load_scores() got an unexpected keyword argument 'datasets'

In [28]:

datasets = ["german_credit"]#,"toy_dataset"]#"adult48k"
cf_methods = ["ils","ils_latent","lore","dice"]
models = ["random_forest","mlp","xgboost","lgbm"]
# Build the analysis dataframe
from check_stast import load_scores

In [47]:
# now let's see if there is a best method for each model
super_df = pd.DataFrame()
for d in datasets:
    for c_m in cf_methods:
        fname = os.path.join("results","subpartial_yet_easier_results_")
        fname += d + "_" + c_m + ".pkl"
        df = pickle.load(open(fname,"rb"))
        print(df.keys())
        for m in models:
            print(d,c_m,m)
            print(len(df[m]["df"].columns))
            print(df[m]["df"].columns[:3],df[m]["df"].columns[-3:])
            df[m]["df"]["model"] = m
            df[m]["df"]["dataset"] = d
            df[m]["df"]["cf_method"] = c_m
            df[m]["df"].columns = [dfc.split("_")[-1] for dfc in df[m]["df"].columns]
            super_df = super_df.append(df[m]["df"])
            print(super_df.shape)
            

dict_keys(['random_forest', 'mlp', 'xgboost', 'lgbm'])
german_credit ils random_forest
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object') Index(['ils_0.09', 'ils_0.04', 'ils_0.0'], dtype='object')
(62, 35)
german_credit ils mlp
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object') Index(['ils_0.09', 'ils_0.04', 'ils_0.0'], dtype='object')
(124, 35)
german_credit ils xgboost
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object') Index(['ils_0.09', 'ils_0.04', 'ils_0.0'], dtype='object')
(186, 35)
german_credit ils lgbm
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object') Index(['ils_0.09', 'ils_0.04', 'ils_0.0'], dtype='object')
(248, 35)
dict_keys(['random_forest', 'mlp', 'xgboost', 'lgbm'])
german_credit ils_latent random_forest
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object') Index(['ils_0.09', 'ils_0.04', 'ils_0.0'], dtype='object')
(310, 35)
german_credit ils_latent mlp
32
Index(['ils_0.99', 'ils_0.96', 'ils_0.95'], dtype='object

In [51]:
from selective_classifiers import visualize_results

ModuleNotFoundError: No module named 'selective_classifiers'

In [ ]:
# group by dataset, model, and plot 
for dt_name,gr in super_df.groupby("dataset"):
    for m_name,gr_m in gr.groupby("model"):
        print(dt_name,m_name)
        
        

german_credit lgbm
                                        0.99      0.96      0.95      0.91  \
PlugInRule                             0.744  0.746988  0.755102  0.754098   
PlugInRuleAUC                          0.744  0.745968  0.743802  0.743802   
CFDistRejector_inf_min                 0.760  0.762626  0.765625  0.768421   
CFDistRejector_inf_min_gamma           0.760  0.768421  0.762712  0.762712   
CFDistRejector_inf_max                 0.760  0.757576  0.755208  0.755208   
...                                      ...       ...       ...       ...   
CFDistRejector_sqeuclidean_min_gamma   0.760  0.758794  0.760417  0.760417   
CFDistRejector_sqeuclidean_max         0.760  0.757576  0.755208  0.757895   
CFDistRejector_sqeuclidean_max_gamma   0.760  0.758794  0.752577  0.755208   
CFDistRejector_sqeuclidean_mean        0.760  0.757576  0.750000  0.757895   
CFDistRejector_sqeuclidean_mean_gamma  0.760  0.756345  0.750000  0.757895   

                                            